# FIMsens — Full Pipeline Example

**Case study:** Lake Powell, Utah/Arizona — post-rainfall SAR flood event (2020-09-17 to 2020-09-20)

| Stage | What happens |
|---|---|
| **1. Download** | Search GEE → preview scenes on an interactive map → select images → choose JRC & DEM → download |
| **2. Raw mapping** | Lee filter → adaptive tiling → bimodal threshold fitting → IDW classification → region grow → cleanup |
| **3. Commission error filter** | Label flood objects → compute attributes → rule-based false-positive removal |
| **4. Depth estimation** | LWSE estimation → IDW interpolation → region-grow depth map |

---
**Before you start**

**Setup** (see the repository README for details)
```bash
conda activate fimsens
pip install -e . --no-deps
```

**Downloaded pixel units**  
- Sentinel-1 — SAR backscatter σ₀ in **dB**  
- Sentinel-2 — surface reflectance **0–1**  
- Landsat 8/9 — surface reflectance **0–1**  
- JRC water — occurrence frequency **0–100 %**  
- DEM — elevation in **metres**

## 0. Global configuration

Edit paths and parameters here before running anything else.

In [19]:
import sys, os

# ════════════════════════════════════════════════════════════════════════════
# PACKAGE
# ════════════════════════════════════════════════════════════════════════════
# fimsens is expected to be installed in the active environment:
#
#     conda env create -f environment.yml
#     conda activate fimsens
#     pip install -e . --no-deps      # from the repo root
#
# The line below both imports it and prints WHICH copy is being used — worth
# checking if you have more than one environment or checkout.
import fimsens
print(f'fimsens {fimsens.__version__} loaded from {os.path.dirname(fimsens.__file__)}')

# ════════════════════════════════════════════════════════════════════════════
# GEE project
# ════════════════════════════════════════════════════════════════════════════
GEE_PROJECT = 'ee-tiandan-gee'   # e.g. 'ee-yourname'

# ── Study area (WGS-84 bounding box) ─────────────────────────────────────────
# Lake Powell flood event, Utah/Arizona
AOI_BBOX      = [94.3982728, 26.7125512, 94.9426219, 27.2754671]   # [xmin, ymin, xmax, ymax]
AOI_SHAPEFILE = None   # alternative: r'path/to/aoi.shp'

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = '2026-07-17'
END_DATE   = '2026-07-29'
MAX_CLOUD  = 30   # max cloud cover % for S2 and Landsat (ignored for S1)

# ── Root output directory ─────────────────────────────────────────────────────
ROOT = r'output/india_flood_2026'   # e.g. r'output/my_flood_event'

DIR_DOWNLOAD = os.path.join(ROOT, '01_download')
DIR_RAW      = os.path.join(ROOT, '02_raw')
DIR_FILTER   = os.path.join(ROOT, '03_filter')
DIR_GROW     = os.path.join(ROOT, '04_grow')

for d in [DIR_DOWNLOAD, DIR_RAW, DIR_FILTER, DIR_GROW]:
    os.makedirs(d, exist_ok=True)

print('Output folders ready.')
AOI = AOI_SHAPEFILE if AOI_SHAPEFILE else AOI_BBOX


fimsens loaded from: D:\AI_Projects\FIMtoolset\fimsens
Output folders ready.


---
## Stage 1 — Download

### 1.1 Authenticate and initialise GEE

In [20]:
import ee
import ipywidgets as widgets
from IPython.display import display

from fimsens.download import GEESearcher, GEEDownloader, DEM_CATALOG, parse_aoi
from fimsens.download.gee_preview import GEEPreview   # not re-exported; import directly

# Run ee.Authenticate() once to store credentials, then comment it out.
# ee.Authenticate()

ee.Initialize(project=GEE_PROJECT)

ee_geom, extent_gcs = parse_aoi(AOI)
print(f'GEE initialised.')
print(f'AOI extent : {extent_gcs}')
print(f'Date range : {START_DATE}  →  {END_DATE}')

GEE initialised.
AOI extent : 94.3982728,26.7125512,94.9426219,27.2754671
Date range : 2026-07-17  →  2026-07-29


### 1.2 Search for available imagery

`search_all` queries Sentinel-1, Sentinel-2, and Landsat 8/9 simultaneously. Results are returned as a dict with keys `'s1'`, `'s2'`, `'landsat'`.

In [21]:
searcher = GEESearcher()
results  = searcher.search_all(AOI, START_DATE, END_DATE, max_cloud=MAX_CLOUD)

Searching Sentinel-1  (2026-07-17 → 2026-07-29) …
  → 4 scenes found
Searching Sentinel-2  (cloud ≤ 30%) …
  → 0 scenes found
Searching Landsat 8/9 (cloud ≤ 30%) …
  → 0 scenes found

Total: 4 scenes


In [22]:
# Display all found scenes as a colour-coded table
# Blue = S1, Green = S2, Orange = Landsat
df = GEESearcher.to_dataframe(results)

if df.empty:
    print('No images found. Try widening the date range or increasing MAX_CLOUD.')
else:
    sensor_colours = {'S1': '#d6eaf8', 'S2': '#d5f5e3', 'Landsat': '#fdebd0'}
    def _row_colour(row):
        c = sensor_colours.get(row['sensor'], '#ffffff')
        return [f'background-color: {c}'] * len(row)
    display(df.style.apply(_row_colour, axis=1))

,sensor,image_id,date,orbitProperties_pass,relativeOrbitNumber_start,platform_number
idx,,,,,,
0,S1,COPERNICUS/S1_GRD/S1D_IW_GRDH_1SDV_20260717T233756_20260717T233821_003720_006A9E_7B2A,2026-07-17,DESCENDING,4,D
1,S1,COPERNICUS/S1_GRD/S1D_IW_GRDH_1SDV_20260722T113958_20260722T114023_003786_006CD7_FA7E,2026-07-22,ASCENDING,70,D
2,S1,COPERNICUS/S1_GRD/S1D_IW_GRDH_1SDV_20260722T114023_20260722T114048_003786_006CD7_6A41,2026-07-22,ASCENDING,70,D
3,S1,COPERNICUS/S1_GRD/S1D_IW_GRDH_1SDV_20260727T114810_20260727T114835_003859_006F53_EAA2,2026-07-27,ASCENDING,143,D


### 1.3 Preview on interactive map

All scenes are added as toggleable layers (off by default).  
- SAR layers: **R = VV  G = VH  B = VH** — water is dark, flooded vegetation is bright green  
- Optical layers: **NIR – Red – Green** false colour — vegetation is red, water is dark blue  

Use the **Layers panel** (top-right of the map) to turn individual scenes on/off.

In [23]:
preview = GEEPreview()
m = preview.create_map(results, aoi_input=AOI, zoom=9)
m

Map(center=[26.99400915, 94.67044735], controls=(WidgetControl(options=['position', 'transparent_bg'], positio…

In [ ]:
# Optional: overlay JRC permanent water and/or a DEM on the map for visual context.
# Uncomment whichever layers you want.

# jrc_img = searcher.get_jrc()
# preview.add_jrc(jrc_img)

# dem_img = searcher.get_dem('SRTM 30m', ee_geom)
# preview.add_dem(dem_img, 'SRTM 30m')

### 1.4 Select images to download

Tick the scenes you want. Use **Select all** / **Clear all** or tick individually.  
For SAR flood mapping, select the S1 scene **closest to the flood peak**.  
If multiple ascending/descending passes are available, prefer the pass with the best look angle over your AOI.

In [13]:
sel_widget = preview.create_selection_widget(results)
display(sel_widget)

In [14]:
# Confirm which scenes are selected before downloading
selected_records = preview.get_selected_records(results)
print(f'{len(selected_records)} image(s) selected:')
for r in selected_records:
    print(f"  [{r['sensor']}]  {r['date']}  {r['image_id'].split('/')[-1]}")

0 image(s) selected:


### 1.5 Choose auxiliary data — JRC & DEM

The JRC permanent-water mask is used as the region-grow seed in Stage 2.  
The DEM is required for Stage 3 (object attributes, depression analysis) and Stage 4 (depth estimation).

| DEM | Coverage | Resolution | Notes |
|---|---|---|---|
| SRTM 30m | Global | 30 m | Default; void-filled |
| NASADEM 30m | Global | 30 m | Improved SRTM reprocessing |
| Copernicus GLO-30 | Global | 30 m | Best global accuracy; recommended |
| ALOS AW3D30 30m | Global | 30 m | DSM (includes vegetation canopy) |
| USGS 3DEP 10m | CONUS + HI + AK | 10 m | Highest res for USA |
| USGS 3DEP 1m | CONUS (sparse) | 1 m | Lidar-derived; large download |

In [15]:
w_jrc = widgets.ToggleButton(
    value=True,
    description='Download JRC Water',
    button_style='info',
    icon='tint',
    layout=widgets.Layout(width='220px'),
)

w_dem = widgets.RadioButtons(
    options=['None'] + list(DEM_CATALOG.keys()),
    value='Copernicus GLO-30',   # recommended default
    description='DEM source:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px'),
)

_dem_info = {
    'None':              'Skip DEM download',
    'SRTM 30m':          'USGS SRTM — 30 m — global, void-filled',
    'NASADEM 30m':       'NASA NASADEM — 30 m — improved SRTM reprocessing',
    'ALOS AW3D30 30m':   'JAXA ALOS AW3D30 — 30 m — DSM (surface, includes vegetation)',
    'Copernicus GLO-30': 'Copernicus DEM GLO-30 — 30 m — EarthDEM-derived, best global quality',
    'USGS 3DEP 10m':     'USGS 3DEP — 10 m — CONUS + Hawaii + Alaska seamless mosaic',
    'USGS 3DEP 1m':      'USGS 3DEP — 1 m — lidar-derived, CONUS only (sparse collection)',
}

w_dem_info = widgets.HTML()
def _update_info(change):
    w_dem_info.value = f"<i style='color:#555'>{_dem_info.get(change['new'], '')}</i>"
w_dem.observe(_update_info, names='value')
_update_info({'new': w_dem.value})

display(widgets.VBox([
    widgets.HTML("<h4 style='margin:4px 0'>Auxiliary data</h4>"),
    w_jrc,
    widgets.HTML("<br>"),
    w_dem,
    w_dem_info,
]))

### 1.6 Configure download settings

| Parameter | Value | Rationale |
|---|---|---|
| `SCALE_S1` | `10` | Sentinel-1 IW GRD native resolution |
| `TILE_SIZE` | `1000` | Pixels per download tile. Reduce to 512 if GEE returns a "request size too large" error |
| `INNER_THREADS` | `10` | Concurrent tile downloads. Reduce on slow connections |

In [16]:
# Resolution per sensor (metres)
SCALE_S1      = 10    # Sentinel-1  native: 10 m
SCALE_S2      = 10    # Sentinel-2  native: 10 m
SCALE_LANDSAT = 30    # Landsat     native: 30 m
scale_map = {'S1': SCALE_S1, 'S2': SCALE_S2, 'Landsat': SCALE_LANDSAT}

# Download engine settings
TILE_SIZE       = 1000     # pixels per tile
INNER_THREADS   = 10       # concurrent download threads
BLOCK_THRESHOLD = 10_000   # pixel threshold for sub-tiling

print('Download configuration:')
print(f'  Output folder   : {DIR_DOWNLOAD}')
print(f'  Scale map       : {scale_map}')
print(f'  Tile size       : {TILE_SIZE} px')
print(f'  Threads         : {INNER_THREADS}')

Download configuration:
  Output folder   : output/lake_powell\01_download
  Scale map       : {'S1': 10, 'S2': 10, 'Landsat': 30}
  Tile size       : 1000 px
  Threads         : 10


### 1.7 Review download plan and execute

In [17]:
# Refresh selected records (in case the widget was changed after cell 1.4)
selected_records = preview.get_selected_records(results)

print('=== Download plan ===')
print(f'  Satellite images : {len(selected_records)}')
for r in selected_records:
    sc = scale_map.get(r['sensor'], 10)
    print(f"    [{r['sensor']}]  {r['date']}  @ {sc} m")
print(f'  JRC water        : {w_jrc.value}')
print(f'  DEM              : {w_dem.value}')
print(f'  Output folder    : {DIR_DOWNLOAD}')

if not selected_records and not w_jrc.value and w_dem.value == 'None':
    print('\n⚠  Nothing selected – nothing to download.')

=== Download plan ===
  Satellite images : 0
  JRC water        : True
  DEM              : Copernicus GLO-30
  Output folder    : output/lake_powell\01_download


In [ ]:
dl = GEEDownloader(
    workspace       = DIR_DOWNLOAD,
    extent_gcs      = extent_gcs,
    target_scale    = SCALE_S1,
    tile_size       = TILE_SIZE,
    inner_threads   = INNER_THREADS,
    block_threshold = BLOCK_THRESHOLD,
)

if selected_records:
    print(f'Downloading {len(selected_records)} satellite image(s) …')
    dl.download_images(selected_records, scale_map=scale_map)
else:
    print('No satellite images selected.')

if w_jrc.value:
    jrc_path = dl.download_jrc(scale=30)

if w_dem.value != 'None':
    dem_path = dl.download_dem(dem_name=w_dem.value, ee_geometry=ee_geom)

print(f'\n✓ All downloads complete.  Files saved to: {DIR_DOWNLOAD}')

In [ ]:
# Locate downloaded files for subsequent stages
# File naming convention: CaseS1_YYYYMMDD_<scene-id>.tif
import glob

s1_files = glob.glob(os.path.join(DIR_DOWNLOAD, 'S1_*.tif'))
assert len(s1_files) >= 1, 'S1 file not found — check DIR_DOWNLOAD'
S1_PATH = s1_files[0]

# jrc_path and dem_path were set during download above.
# If you are re-running from this cell without re-downloading, set them manually:
# jrc_path = os.path.join(DIR_DOWNLOAD, 'CaseJRC_water_occurrence.tif')
# dem_path  = os.path.join(DIR_DOWNLOAD, 'CaseDEM_Copernicus_GLO-30.tif')

print(f'S1  : {S1_PATH}')
print(f'JRC : {jrc_path}')
print(f'DEM : {dem_path}')

**Output file naming reference**

| Dataset | Filename pattern |
|---|---|
| Sentinel-1 | `S1_YYYYMMDD_<scene-id>.tif` |
| Sentinel-2 | `S2_YYYYMMDD_<scene-id>.tif` |
| Landsat 8/9 | `Landsat_YYYYMMDD_<scene-id>.tif` |
| JRC water | `CaseJRC_water_occurrence.tif` |
| DEM | `CaseDEM_<source>.tif` |

All files are reprojected to the UTM zone of the AOI centroid.

---
## Stage 2 — Raw Flood Mapping

```
S1.tif  →  Lee filter  →  Tile split  →  Bimodal fit  →  Outlier filter
        →  IDW classify  →  Region grow  →  Post-process  →  flood_clean.tif
```

### 2.1 Lee speckle filter

| Parameter | Value | Rationale |
|---|---|---|
| `window_size` | `7` | 7×7 window is standard for Sentinel-1 IW GRD |
| `enl` | `4.9` | Equivalent Number of Looks for S1 IW GRD. Compute as (mean/std)² if using a different product |
| `db_input` | `True` | GEE exports S1 in **dB** — this flag converts dB→linear internally before filtering, then converts back |
| `enhanced` | `False` | Standard Lee. Set `True` for Enhanced Lee (better edge preservation, slightly more noise in uniform areas) |

In [ ]:
from fimsens.raw import leeFilt

S1_LEE = os.path.join(DIR_RAW, 'S1_lee.tif')

leeFilt(
    input_path  = S1_PATH,
    output_path = S1_LEE,
    window_size = 7,      # must be odd; 7 is standard for S1 IW GRD
    enl         = 4.9,    # Equivalent Number of Looks for S1 IW GRD
    enhanced    = False,
    db_input    = True,   # GEE exports S1 in dB
)

### 2.2 Adaptive tile splitting

| Parameter | Value | Rationale |
|---|---|---|
| `threshold` | `0.3` | Select tiles where `0.3 < mean < 0.7` (mixed water/land content). Increase to `0.4` in dry scenes |
| `tile_size` | `5000` | Initial tile side in metres. `2000–5000 m` is typical |
| `levels` | `4` | Quadtree recursion depth (each level halves tile size) |

In [ ]:
from fimsens.raw import region_split

region_split(
    workspace        = DIR_RAW,
    water_image_path = S1_LEE,
    threshold        = 0.3,
    tile_size        = 5000,
    levels           = 4,
)

### 2.3 Bimodal gamma threshold fitting

| Parameter | Value | Rationale |
|---|---|---|
| `gamma` | `'18 31 1 2'` | Initial EM parameters `alpha1 alpha2 rate1 rate2`, calibrated for SAR **dB** values (−30 to 0 dB). EM refines these over 100 iterations |
| `image_type` | `'SAR'` | Use `'optical'` for NIR reflectance [0–1] |

In [ ]:
from fimsens.raw import bimodalFit

bimodalFit(
    workspace  = DIR_RAW,
    sarImage   = S1_LEE,
    regions    = os.path.join(DIR_RAW, 'regions.shp'),
    gamma      = '18 31 1 2',
    image_type = 'SAR',
)

### 2.4 Outlier threshold filtering

| Parameter | Value | Rationale |
|---|---|---|
| `BC` | `0.555` | Bimodality Coefficient cutoff. Tiles with BC > 0.555 are clearly bimodal and get a tighter outlier filter (±2σ vs ±1.5σ) |

In [ ]:
from fimsens.raw import filterOutliers

filterOutliers(
    workspace        = DIR_RAW,
    threshold_points = os.path.join(DIR_RAW, 'threshold.shp'),
    BC               = 0.555,
)

### 2.5 IDW flood classification

| Parameter | Value | Rationale |
|---|---|---|
| `power` | `2.0` | IDW exponent. Higher values (3–4) → more localised threshold surface. `2.0` is standard |

In [ ]:
from fimsens.raw import interpFlood

interpFlood(
    workspace        = DIR_RAW,
    threshold_points = os.path.join(DIR_RAW, 'threshold_filtered.shp'),
    field            = 'threshold',
    sarImage         = S1_LEE,
    power            = 2.0,
)

### 2.6 Region grow from JRC seed

| Parameter | Value | Rationale |
|---|---|---|
| `threshold` | `-15` | Growth threshold in **dB** (VV). Pixels below this that are 8-connected to a JRC seed pixel are classified as flood. If too little water: raise (e.g., −13). If over-growing: lower (e.g., −18). For VH use −20 to −23 dB |

In [ ]:
from fimsens.raw import regionGrow

regionGrow(
    workspace       = DIR_RAW,
    seed_image_path = jrc_path,
    grow_image_path = S1_LEE,
    threshold       = -15,   # dB grow threshold for VV; adjust per scene
)

### 2.7 Morphological post-processing

| Parameter | Value | Rationale |
|---|---|---|
| `minObject` | `100` | At 10 m: 100 px = 1 ha. Increase to 500 in urban areas |
| `minHoleSize` | `10000` | At 10 m: 10,000 px = 1 km². Holes larger than this (islands) are preserved |

In [ ]:
from fimsens.raw import postProcess

FLOOD_CLEAN = os.path.join(DIR_RAW, 'flood_clean.tif')

postProcess(
    inImg       = os.path.join(DIR_RAW, 'water_connect.tif'),
    outImg      = FLOOD_CLEAN,
    minObject   = 100,
    minHoleSize = 10000,
)

### 2.8 Visualise raw flood map

In [ ]:
import rasterio, numpy as np, matplotlib.pyplot as plt

files = {
    'S1 Lee filtered (dB)':  (S1_LEE,                                          'RdBu'),
    'IDW threshold surface': (os.path.join(DIR_RAW, 'threshold_idw.tif'),       'RdYlGn'),
    'Raw flood map':         (os.path.join(DIR_RAW, 'flood.tif'),               'Blues'),
    'Cleaned flood map':     (FLOOD_CLEAN,                                       'Blues'),
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, (fpath, cmap)) in zip(axes, files.items()):
    if not os.path.exists(fpath):
        ax.set_title(f'{title}\n(not found)'); ax.axis('off'); continue
    with rasterio.open(fpath) as src:
        data = src.read(1).astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
    ax.imshow(data, cmap=cmap, interpolation='none')
    ax.set_title(title); ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(DIR_RAW, 'raw_flood_overview.png'), dpi=150)
plt.show()

---
## Stage 3 — Commission Error Filter

| predict | Meaning |
|---|---|
| 0 | False positive — too small |
| 1 | True flood — large confirmed object |
| 2 | Depression-stored water (retained) |
| 4 | False positive — LWSE too high |
| 5 | Tentative flood — LWSE too low |
| 6 | False positive — too far from confirmed flood |
| 7 | True flood — proximate to confirmed flood |

### 3.1 Generate labeled flood objects

| Parameter | Value | Rationale |
|---|---|---|
| `morph_operation` | `'eded'` | Smooths boundaries before labeling; preserves object area |
| `min_back` | `100` | Fill interior holes ≤ 100 px (e.g., gaps from SAR layover) |
| `connectivity` | `2` | 8-connected; better captures continuous water bodies |

In [ ]:
from fimsens.filter import generate_flood_objects

FIM_OBJECTS = os.path.join(DIR_FILTER, 'fim_objects.tif')

fim_objs, meta, n_objects = generate_flood_objects(
    fim_path        = FLOOD_CLEAN,
    output_path     = FIM_OBJECTS,
    min_front       = 0,
    min_back        = 100,
    morph_operation = 'eded',
    connectivity    = 2,
)
print(f'{n_objects} flood objects labeled.')

### 3.2 Compute object attributes and build reference layer

In [ ]:
import rasterio
from fimsens.filter import (
    compute_object_attributes,
    add_depression_attributes,
    ObjectFilter,
    build_reference_layer,
)

with rasterio.open(FIM_OBJECTS) as src:
    t = src.transform
pix_area_m2  = abs(t.a * t.e)
pix_area_km2 = pix_area_m2 / 1e6

# 3.2a — Morphometric + elevation attributes (area, centroid, LWSE, …)
print('Computing object attributes …')
attrs_df = compute_object_attributes(
    label_arr      = fim_objs,
    meta           = meta,
    dem_path       = dem_path,
    pixel_area_km2 = pix_area_km2,
)

# 3.2b — Depression fill + depth/volume attributes (has_pit, depre_vol, depth_vol)
# Required by Rule 2 (depression-stored water)
print('Adding depression attributes …')
attrs_df = add_depression_attributes(
    attrs_df    = attrs_df,
    label_arr   = fim_objs,
    dem_path    = dem_path,
    pix_area_m2 = pix_area_m2,
    output_dir  = os.path.join(DIR_FILTER, 'depression'),
)

attrs_csv = os.path.join(DIR_FILTER, 'object_attributes.csv')
attrs_df.to_csv(attrs_csv, index=False)
print(f'Saved → {attrs_csv}')
attrs_df.head()

In [ ]:
# 3.2c — Identify large objects (Rule 0 + Rule 1) to seed the reference LWSE layer
filt_ref = ObjectFilter(attrs_df)
filt_ref.rule0_remove_small(threshold_px=50).rule1_identify_large(n_clusters=2, n_init=50)
filt_ref.summary()

LARGE_OBJECTS = os.path.join(DIR_FILTER, 'large_objects.tif')
filt_ref.save_large_objects(fim_objs, LARGE_OBJECTS, meta)

In [ ]:
# 3.2d — Build LWSE reference layer from large-object boundaries
# n_points=200: boundary pixels per segment (KDE → one LWSE per segment)
# k_neighbors=5: IDW outlier replacement within the reference point field
reference_df = build_reference_layer(
    large_obj_path = LARGE_OBJECTS,
    dem_path       = dem_path,
    output_dir     = os.path.join(DIR_FILTER, 'reference'),
    n_points       = 200,
    min_pixels     = 20,
    k_neighbors    = 5,
)
reference_df.to_csv(os.path.join(DIR_FILTER, 'reference.csv'), index=False)
print(f'{len(reference_df)} reference segments.')

### 3.3 Run the full filter pipeline

In [ ]:
filt = ObjectFilter(attrs_df)

(
    filt
    # Rule 0: objects < 50 px → predict=0 (noise)
    .rule0_remove_small(threshold_px=50)

    # Rule 1: KMeans log-area clustering → larger cluster → predict=1 (confirmed flood)
    .rule1_identify_large(n_clusters=2, n_init=50)

    # Rule 2: depression-water objects (has_pit=1 and depth_vol < depre_vol) → predict=2
    .rule2_depression_water()

    # Rule 3: LWSE consistency — auto threshold = mean ± 2σ of reference delta_LWSE
    # Objects too high → predict=4; too low → predict=5 (tentative)
    .rule3_filter_by_delta_lwse(
        reference_df   = reference_df,
        threshold_high = None,   # auto: max(mean + 2σ, 5.0 m)
        threshold_low  = None,   # auto: mean − 2σ
        k_neighbors    = 5,
        min_threshold  = 5.0,
    )

    # Rule 4: distance from confirmed flood — auto KMeans near/far split
    # Far objects → predict=6; near objects → predict=7 (true flood)
    .rule4_filter_by_distance(distance_threshold='auto', n_init=50)
)

filt.summary()

In [ ]:
# Save per-object predict table
filt.save_results(DIR_FILTER, filename_prefix='object_predict')

# Save filtered flood raster — retain predict ∈ {1, 2, 5, 7}
results_df = filt.get_results()
true_ids   = results_df.loc[results_df['predict'].isin([1, 2, 5, 7]), 'ID'].values

import numpy as np
flood_filtered = np.where(np.isin(fim_objs, true_ids), 1, 0).astype(np.uint8)

FLOOD_FILTERED = os.path.join(DIR_FILTER, 'flood_filtered.tif')
out_meta = meta.copy()
out_meta.update(dtype='uint8', count=1, nodata=255)
with rasterio.open(FLOOD_FILTERED, 'w', **out_meta) as dst:
    dst.write(flood_filtered, 1)
print(f'Filtered flood raster → {FLOOD_FILTERED}')

---
## Stage 4 — Flood Depth Estimation

### 4.1 Generate LWSE layers

| Parameter | Value | Rationale |
|---|---|---|
| `n_points` | `200` | Boundary pixels per LWSE segment. Fewer → finer resolution but noisier |
| `min_pixels` | `20` | Skip objects with fewer than 20 boundary pixels |
| `k_neighbors` | `5` | Neighbours for IDW outlier replacement in the LWSE point field |

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

from fimsens.grow import generate_lwse_layers

BOUNDARY_SHP = os.path.join(DIR_GROW, 'boundary_elev.shp')
CENTROID_SHP = os.path.join(DIR_GROW, 'centroid_elev.shp')

generate_lwse_layers(
    input_image          = FLOOD_FILTERED,
    dem_path             = dem_path,
    n_points             = 200,
    min_pixels           = 20,
    k_neighbors          = 5,
    output_boundary_path = BOUNDARY_SHP,
    output_centroid_path = CENTROID_SHP,
)

### 4.2 IDW depth mapping with region grow

| Parameter | Value | Rationale |
|---|---|---|
| `fieldName` | `'LWSE_new'` | Outlier-filtered LWSE from Step 4.1 |
| `max_points` | `12` | IDW neighbourhood size (8–15 practical range) |
| `power` | `3.0` | IDW exponent. Higher → more localised surface. `3.0` balances local fidelity and spatial continuity |

**Classification output:**  0 = non-flood, 1 = seed flood, 2 = grown flood (DEM < LWSE)

In [ ]:
from fimsens.grow import idw2Flood1

FLOOD_DEPTH = os.path.join(DIR_GROW, 'FloodDepth.tif')
FLOOD_CLASS = os.path.join(DIR_GROW, 'FloodClass.tif')

idw2Flood1(
    fcName            = CENTROID_SHP,
    fieldName         = 'LWSE_new',
    demName           = dem_path,
    floodSeedRaster   = FLOOD_FILTERED,
    idw_temp          = os.path.join(DIR_GROW, 'idw_temp.tif'),
    outRaster         = FLOOD_DEPTH,
    outClassRaster    = FLOOD_CLASS,
    output_shapefile  = os.path.join(DIR_GROW, 'reprojected.shp'),
    alignedSeedRaster = os.path.join(DIR_GROW, 'aligned_seed.tif'),
    max_points        = 12,
    power             = 3.0,
)

### 4.3 Visualise depth and classification

In [ ]:
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

with rasterio.open(FLOOD_FILTERED) as src:
    flood_arr = src.read(1).astype(float)
    flood_arr[flood_arr == src.nodata] = np.nan
axes[0].imshow(flood_arr, cmap='Blues', interpolation='none')
axes[0].set_title('Filtered flood mask (Stage 3)'); axes[0].axis('off')

with rasterio.open(FLOOD_CLASS) as src:
    class_arr = src.read(1).astype(float)
cmap_class = mcolors.ListedColormap(['white', '#2196F3', '#00BCD4'])
axes[1].imshow(class_arr, cmap=cmap_class, vmin=0, vmax=2, interpolation='none')
axes[1].set_title('Flood class\n(blue=seed, cyan=grown)'); axes[1].axis('off')

with rasterio.open(FLOOD_DEPTH) as src:
    depth_arr = src.read(1).astype(float)
    depth_arr[~np.isfinite(depth_arr)] = np.nan
im = axes[2].imshow(depth_arr, cmap='YlOrRd', interpolation='none')
fig.colorbar(im, ax=axes[2], label='Depth (m)', shrink=0.8)
axes[2].set_title('Flood depth (m)'); axes[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(DIR_GROW, 'flood_depth_overview.png'), dpi=150)
plt.show()

valid_depth = depth_arr[np.isfinite(depth_arr)]
flood_km2   = float(np.nansum(flood_arr > 0)) * pix_area_km2
print(f'Flood area       : {flood_km2:.2f} km²')
print(f'Mean flood depth : {valid_depth.mean():.2f} m')
print(f'Max flood depth  : {valid_depth.max():.2f} m')

---
## Output summary

| File | Description |
|---|---|
| `01_download/S1_*.tif` | Sentinel-1 SAR backscatter (dB) |
| `01_download/CaseJRC_*.tif` | JRC permanent water mask |
| `01_download/CaseDEM_*.tif` | Digital Elevation Model |
| `02_raw/S1_lee.tif` | Lee-filtered SAR image (dB) |
| `02_raw/flood_clean.tif` | Raw flood binary map |
| `03_filter/object_predict.csv` | Per-object predict labels |
| `03_filter/flood_filtered.tif` | Commission-error-filtered flood mask |
| `04_grow/FloodClass.tif` | 3-class raster (0=dry, 1=seed, 2=grown) |
| `04_grow/FloodDepth.tif` | Flood depth in metres |

For full parameter guidance, see `README_v2.md`.